In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

performance_df = pd.read_excel(file_name)

print(performance_df.shape)
performance_df.head()

Saving Performance_Dataset_June2026.xlsx to Performance_Dataset_June2026 (11).xlsx
(4217, 17)


,task_id,task_code,task_name,wbs_id,target_start_date,target_end_date,baseline_duration_days,total_float_hr_cnt,planned_percent,actual_percent,progress_variance,budget_cost,PV,EV,AC,SPI,CPI
0,143649,HCSWSSZ1000,SS Z1 :Excavation Works - P1,25860,2012-06-05 08:00:00,2012-06-20 16:00:00,15,16,100,86,-14,24390,24390,20975.40,24999.573676,0.86,0.839030
1,143650,HCSWSSZ1010,SS Z1 :Pour blinding under membrane,25880,2012-06-10 08:00:00,2012-06-23 16:00:00,13,16,100,99,-1,25467,25467,25212.33,30639.188589,0.99,0.822879
2,143651,HMMOAZ000,Erect Tower crane # 1,25702,2012-06-07 08:00:00,2012-06-21 16:00:00,14,296,100,100,0,19040,19040,19040.00,16579.277586,1.00,1.148422
3,143652,HMMOAZ010,Erect Tower crane # 2,25702,2012-07-01 08:00:00,2012-07-23 16:00:00,22,240,100,94,-6,39468,39468,37099.92,36565.029291,0.94,1.014628
4,143653,HMMOAZ020,Erect Tower crane # 3,25702,2012-07-28 08:00:00,2012-08-22 16:00:00,25,112,100,90,-10,40750,40750,36675.00,39691.877975,0.90,0.923993


In [ ]:
performance_df.columns

Index(['task_id', 'task_code', 'task_name', 'wbs_id', 'target_start_date',
       'target_end_date', 'baseline_duration_days', 'total_float_hr_cnt',
       'planned_percent', 'actual_percent', 'progress_variance', 'budget_cost',
       'PV', 'EV', 'AC', 'SPI', 'CPI'],
      dtype='object')

In [ ]:
performance_df["cost_issue"] = (
    performance_df["CPI"] < 0.90
)

In [ ]:
performance_df["progress_issue"] = (
    performance_df["progress_variance"] < -10
)

In [ ]:
performance_df["schedule_issue"] = (
    performance_df["SPI"] < 0.90
)

In [ ]:
performance_df["float_days"] = (
    pd.to_numeric(
        performance_df["total_float_hr_cnt"],
        errors="coerce"
    ) / 8
)

performance_df["float_issue"] = (
    performance_df["float_days"] <= 5
)

In [ ]:
print(
    "Schedule Issues:",
    performance_df["schedule_issue"].sum()
)

print(
    "Cost Issues:",
    performance_df["cost_issue"].sum()
)

print(
    "Progress Issues:",
    performance_df["progress_issue"].sum()
)

print(
    "Float Issues:",
    performance_df["float_issue"].sum()
)

Schedule Issues: 1373
Cost Issues: 2266
Progress Issues: 1367
Float Issues: 524


In [ ]:
performance_df["alert_score"] = (
    performance_df["schedule_issue"].astype(int)
    + performance_df["cost_issue"].astype(int)
    + performance_df["progress_issue"].astype(int)
    + 2 * performance_df["float_issue"].astype(int)
)

performance_df[
    [
        "task_code",
        "SPI",
        "CPI",
        "progress_variance",
        "float_days",
        "alert_score"
    ]
].head()

,task_code,SPI,CPI,progress_variance,float_days,alert_score
0,HCSWSSZ1000,0.86,0.839030,-14,2.0,5
1,HCSWSSZ1010,0.99,0.822879,-1,2.0,3
2,HMMOAZ000,1.00,1.148422,0,37.0,0
3,HMMOAZ010,0.94,1.014628,-6,30.0,0
4,HMMOAZ020,0.90,0.923993,-10,14.0,0


In [ ]:
performance_df["alert_level"] = np.select(
    [
        performance_df["alert_score"] == 0,
        performance_df["alert_score"].between(1, 2),
        performance_df["alert_score"].between(3, 4),
        performance_df["alert_score"] >= 5
    ],
    [
        "Green",
        "Yellow",
        "Orange",
        "Red"
    ],
    default="Unknown"
)

performance_df[
    [
        "task_code",
        "alert_score",
        "alert_level"
    ]
].head(10)

,task_code,alert_score,alert_level
0,HCSWSSZ1000,5,Red
1,HCSWSSZ1010,3,Orange
2,HMMOAZ000,0,Green
3,HMMOAZ010,0,Green
4,HMMOAZ020,0,Green
5,HCSWSSZ1020,5,Red
6,HCSWSSZ1030,2,Yellow
7,HCSWSSZ1040,2,Yellow
8,HCSWSSZ1050,5,Red
9,HCSWSSZ1060,0,Green


In [ ]:
print(
    "Green Activities:",
    (performance_df["alert_level"] == "Green").sum()
)

print(
    "Yellow Activities:",
    (performance_df["alert_level"] == "Yellow").sum()
)

print(
    "Orange Activities:",
    (performance_df["alert_level"] == "Orange").sum()
)

print(
    "Red Activities:",
    (performance_df["alert_level"] == "Red").sum()
)

Green Activities: 1458
Yellow Activities: 1491
Orange Activities: 1136
Red Activities: 132


In [ ]:
performance_df["recommendation"] = np.select(
    [
        performance_df["alert_level"] == "Green",
        performance_df["alert_level"] == "Yellow",
        performance_df["alert_level"] == "Orange",
        performance_df["alert_level"] == "Red"
    ],
    [
        "Continue normal monitoring.",
        "Increase monitoring frequency and investigate performance trends.",
        "Develop recovery plan and review schedule and cost exposure.",
        "Executive escalation required. Immediate recovery actions and management review needed."
    ],
    default="Review activity."
)

performance_df[
    [
        "task_code",
        "alert_score",
        "alert_level",
        "recommendation"
    ]
].head(10)

,task_code,alert_score,alert_level,recommendation
0,HCSWSSZ1000,5,Red,Executive escalation required. Immediate recov...
1,HCSWSSZ1010,3,Orange,Develop recovery plan and review schedule and ...
2,HMMOAZ000,0,Green,Continue normal monitoring.
3,HMMOAZ010,0,Green,Continue normal monitoring.
4,HMMOAZ020,0,Green,Continue normal monitoring.
5,HCSWSSZ1020,5,Red,Executive escalation required. Immediate recov...
6,HCSWSSZ1030,2,Yellow,Increase monitoring frequency and investigate ...
7,HCSWSSZ1040,2,Yellow,Increase monitoring frequency and investigate ...
8,HCSWSSZ1050,5,Red,Executive escalation required. Immediate recov...
9,HCSWSSZ1060,0,Green,Continue normal monitoring.


In [ ]:
top_escalation = (
    performance_df
    .sort_values(
        by="alert_score",
        ascending=False
    )
    [
        [
            "task_code",
            "task_name",
            "SPI",
            "CPI",
            "progress_variance",
            "float_days",
            "alert_score",
            "alert_level",
            "recommendation"
        ]
    ]
    .head(20)
)

top_escalation

,task_code,task_name,SPI,CPI,progress_variance,float_days,alert_score,alert_level,recommendation
26,HCSWSSZ3010,SS Z3 :Pour blinding under membrane - P1,0.85,0.887628,-15,4.0,5,Red,Executive escalation required. Immediate recov...
27,HCSWSSZ3020,SS Z3 :Construct block works (formwork),0.81,0.766292,-19,2.0,5,Red,Executive escalation required. Immediate recov...
30,HCSWSSZ3050,SS Z3 :Cast concrete for foundations,0.80,0.767204,-20,2.0,5,Red,Executive escalation required. Immediate recov...
8,HCSWSSZ1050,SS Z1 :Cast concrete for foundations,0.86,0.745475,-14,2.0,5,Red,Executive escalation required. Immediate recov...
1249,HCFWB3Z1011,B3 Z1 :Paint Final Layer,0.82,0.695244,-18,3.0,5,Red,Executive escalation required. Immediate recov...
1250,HCFWB3Z1004,B3 Z1 :Epoxy Screed Finish Works,0.88,0.727564,-12,3.0,5,Red,Executive escalation required. Immediate recov...
1574,HCFWGFZ3014,GF Z3 :Plaster Final Layer,0.80,0.851208,-20,0.0,5,Red,Executive escalation required. Immediate recov...
1583,HCFWGFZ3021,GF Z3 :Plaster Second Layer,0.88,0.867028,-12,0.0,5,Red,Executive escalation required. Immediate recov...
1614,HCFWGFZ3010,GF Z3 :Gybsum Board Partitions,0.86,0.781259,-14,0.0,5,Red,Executive escalation required. Immediate recov...
1750,HCFWF2Z3069,F2 Z3 :Plaster Final Layer,0.81,0.749215,-19,0.0,5,Red,Executive escalation required. Immediate recov...


In [ ]:
executive_kpis = pd.DataFrame({
    "KPI": [
        "Total Activities",
        "Green Activities",
        "Yellow Activities",
        "Orange Activities",
        "Red Activities"
    ],
    "Value": [
        len(performance_df),
        (performance_df["alert_level"] == "Green").sum(),
        (performance_df["alert_level"] == "Yellow").sum(),
        (performance_df["alert_level"] == "Orange").sum(),
        (performance_df["alert_level"] == "Red").sum()
    ]
})

executive_kpis

,KPI,Value
0,Total Activities,4217
1,Green Activities,1458
2,Yellow Activities,1491
3,Orange Activities,1136
4,Red Activities,132


In [ ]:
with pd.ExcelWriter(
    "Project_Controls_Early_Warning_Report_V01.xlsx"
) as writer:

    executive_kpis.to_excel(
        writer,
        sheet_name="Executive_KPIs",
        index=False
    )

    top_escalation.to_excel(
        writer,
        sheet_name="Top_Escalation_Activities",
        index=False
    )

    performance_df.to_excel(
        writer,
        sheet_name="Early_Warning_Dataset",
        index=False
    )

print(
    "Project Controls Early Warning report exported successfully."
)

Project Controls Early Warning report exported successfully.


In [ ]:
from google.colab import files

files.download(
    "Project_Controls_Early_Warning_Report_V01.xlsx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>